# Быстрая проверка решения

Для запуска нужны три исходных Parquet в `data/` и ядро **Avito ranking**.
Выберите **Run → Run All Cells**. Обучение здесь не повторяется.

Ноутбук заново строит обучающую историю и поисковые индексы, пересчитывает все 2 452 запроса и сравнивает ответ с приложенным CSV. Установка окружения описана в [README](../README.md).

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

# Ограничиваем число потоков для работы на CPU с небольшим объёмом памяти.
os.environ.update({"POLARS_MAX_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "OMP_NUM_THREADS": "2"})

from avito_ranker.config import load_config
from avito_improved.run import run_stage

# Пути работают при запуске из корня репозитория и из папки notebooks.
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    # Пути работают при запуске из корня репозитория и из папки notebooks.
    project_dir = project_dir.parent
# Оба ноутбука используют одни настройки и одни модули расчёта.
config_path = project_dir / "configs/improved.toml"
config = load_config(config_path)
work = config["work_dir"] / "quality"
saved = config["results_dir"]
started = time.perf_counter()
# Сначала запускаем тесты; при ошибке выполнение остановится.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=project_dir, check=True)

## 1. Подготовка

История содержит только обучающие группы. Она связывает похожие запросы с выбранными объявлениями и подкатегориями услуг. Исходные данные проверяются до расчёта.

In [ ]:
# Готовим разбиение, корпус и историю только по обучающим группам.
run_stage("prepare", config_path)
# Загружаем готовую модель и сверяем код, данные и историю со снимком.
run_stage("restore", config_path)

## 2. Поисковые индексы

Строятся BM25-индексы заголовков, описаний и параметров. Итоговый поиск использует только объявления из benchmark.

In [ ]:
# Индексируем только объявления, допустимые в итоговом ответе.
run_stage("benchmark_indices", config_path)

## 3. Кандидаты и признаки

К текстовому поиску добавляются кандидаты по фильтрам и похожим запросам из train. Модель учитывает текст, локацию, расстояние, фильтры и свойства объявления.

In [ ]:
# Повторяем поиск и расчёт признаков для всех запросов benchmark.
run_stage("benchmark_features", config_path)

## 4. Ответ

Сохранённая модель выбирает до 50 объявлений для каждого запроса. Проверяются исходные идентификаторы и формат файла.

In [ ]:
# Применяем сохранённую модель и записываем проверенный CSV.
run_stage("predict", config_path)

## 5. Сравнение с готовым файлом

Успешная проверка заканчивается `status: passed` и `answer_identical: true`.

In [ ]:
from avito_retrieval.submission import validate_answer

# Проверяем формат, полный список запросов и допустимость всех item_id.
computed = work / "answer.csv"
reference = project_dir / "answer.csv"
report = validate_answer(computed, config["data_dir"] / "benchmark_queries.parquet",
                         config["data_dir"] / "benchmark_items.parquet")
# Требуем точного совпадения файлов, включая порядок и переводы строк.
if computed.read_bytes() != reference.read_bytes():
    raise AssertionError("Новый answer.csv отличается от приложенного")
# Успешный отчёт записывается только после всех проверок.
report.update({"status": "passed", "answer_identical": True,
               "seconds": round(time.perf_counter() - started, 2)})
(work / "check_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
report